## JHU Model Transfer to AoU

Applies the trained JHU VaDE model to AoU patients to assign them to the existing JHU clusters.

### Inputs
- **Model code:** `AoU/aou_src/`
- **Trained model:** `outputs/model.pt`
- **JHU reference data:** `data/features_HFCohort_deid.parquet` and `data/meta_HFCohort_deid.parquet`, used to reconstruct preprocessing and feature order.
- **AoU patient features:** `feature_matrix_wide_deaths.csv`
- **AoU feature mapping:** `covariate_id_name_map.csv`

### Outputs
- **Patient assignments:** cluster labels, assignment confidence, and latent coordinates, saved to:
  - `patient_cluster_assignments_jhu_transfer_deaths.parquet`
  - `patient_cluster_assignments_jhu_transfer_deaths.csv`
- **Printed diagnostics:** missing features, cluster counts, mean confidence, silhouette score, and comparisons of JHU and AoU cluster separation, centroid distances, and cluster proportions.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import importlib.util
from sklearn.metrics import silhouette_score, davies_bouldin_score, adjusted_mutual_info_score
from scipy.stats import chi2_contingency
from scipy.spatial.distance import cdist

In [ ]:
def load_module_from_path(module_name, file_path):
    spec = importlib.util.spec_from_file_location(module_name, file_path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module
    
SEED = 42
DEVICE = "cpu"
JHU_REPO = Path("AoU/aou_src").resolve()
jhu_config = load_module_from_path("jhu_config", JHU_REPO / "config.py")
sys.modules["config"] = jhu_config  

jhu_data = load_module_from_path("jhu_data", JHU_REPO / "data.py")
jhu_vade = load_module_from_path("jhu_vade", JHU_REPO / "vade.py")
jhu_inference = load_module_from_path("jhu_inference", JHU_REPO / "inference.py")

preprocess_data = jhu_data.preprocess_data
fit_lab_scaler = jhu_data.fit_lab_scaler
encode_measurements = jhu_data.encode_measurements
merge_unit_splits = jhu_data.merge_unit_splits
remap = jhu_data.remap
neutral_fill = jhu_data.neutral_fill
load_features = jhu_data.load_features

load_checkpoint = jhu_vade.load_checkpoint
predict_clusters = jhu_inference.predict_clusters

### Reconstruct the JHU scaler (median/IQR), band_rate, and feature order 

In [ ]:
JHU_DATA = Path("data")
JHU_OUTPUTS = Path("outputs")

jhu = preprocess_data(
    features_path=str(JHU_DATA / "features_HFCohort_deid.parquet"),
    metadata_path=str(JHU_DATA / "meta_HFCohort_deid.parquet"),
    dataset="heartfailure",
    seed=SEED,
)
jhu_feature_order = list(jhu["feats"].columns)         
ids_train = jhu["index"][jhu["is_train"]]

# fit_lab_scaler
raw_jhu = load_features(str(JHU_DATA / "features_HFCohort_deid.parquet"), impute_zero=True)
jhu_metadata = pd.read_parquet(JHU_DATA / "meta_HFCohort_deid.parquet")
covariate_map = jhu_metadata.set_index("covariateId")["covariateName"].to_dict()
raw_jhu.columns = remap(raw_jhu.columns, covariate_map)
raw_jhu = raw_jhu.loc[:, ~raw_jhu.columns.duplicated()].copy()
jhu_scaler = fit_lab_scaler(raw_jhu.loc[ids_train])
jhu_band_rate = jhu["band_rate"]                       

# load the trained JHU model 
model, payload = load_checkpoint(str(JHU_OUTPUTS / "model.pt"), device=DEVICE)
assert payload["feature_names"] == jhu_feature_order, (
    "Reconstructed feature order doesn't match the checkpoint -- do not proceed, "
    "the encoder input would be silently misaligned."
)

# load AoU features and rename columns 
#aou_features = pd.read_csv("feature_matrix_wide.csv").set_index("person_id")
aou_features = pd.read_csv("feature_matrix_wide_deaths.csv").set_index("person_id")


aou_features = aou_features.loc[:, ~aou_features.columns.duplicated()].copy()

aou_covmap = pd.read_csv("covariate_id_name_map.csv")
aou_name_lookup = aou_covmap.set_index("covariate_id")["remapped_name"].to_dict()
aou_features.columns = [aou_name_lookup.get(int(float(c)), str(c)) for c in aou_features.columns]
aou_features = aou_features.loc[:, ~aou_features.columns.duplicated()].copy()

In [ ]:
missing_scaler_cols = [c for c in jhu_scaler["cols"] if c not in aou_features.columns]
print(f"{len(missing_scaler_cols)} / {len(jhu_scaler['cols'])} JHU measurement columns missing entirely in AoU:")
print(missing_scaler_cols)

for c in missing_scaler_cols:
    aou_features[c] = np.nan


# encode AoU patients using JHU's scaler 
aou_feats, aou_mask = encode_measurements(aou_features, jhu_scaler, clip=4.0)
aou_feats, aou_mask = merge_unit_splits(aou_feats, aou_mask, jhu_metadata)


# align AoU's columns to the JHU model's exact 283-feature order 
missing_in_aou = [c for c in jhu_feature_order if c not in aou_feats.columns]
if missing_in_aou:
    print(f"\n{len(missing_in_aou)} / {len(jhu_feature_order)} JHU features "
          f"have no AoU data at all -- filled from JHU band_rate:")
    print(missing_in_aou)

aou_feats = aou_feats.reindex(columns=jhu_feature_order)
aou_mask = aou_mask.reindex(columns=jhu_feature_order, fill_value=0)
aou_mask = aou_mask.where(aou_feats.notna(), 0).fillna(0)  
aou_feats = aou_feats.fillna(0)                         

# neutral-fill missing entries with the JHU band_rate 
X_aou = aou_feats.to_numpy(dtype="float32")
M_aou = aou_mask.to_numpy(dtype="float32")
X_aou_in = neutral_fill(X_aou, M_aou, jhu_band_rate)


# run every AoU patient through the JHU encoder
aou_assignments = predict_clusters(
    model,
    X_aou_in,
    index=aou_features.index,
    device=DEVICE,
)
aou_assignments.index.name = "person_id"

print(aou_assignments["cluster"].value_counts().sort_index())
print(f"Mean confidence: {aou_assignments['confidence'].mean():.3f}")

In [ ]:
output_path = Path("patient_cluster_assignments_jhu_transfer_deaths.parquet")

output_path.parent.mkdir(parents=True, exist_ok=True)
aou_assignments.reset_index().to_parquet(output_path, index=False)
aou_assignments.reset_index().to_csv(output_path.with_suffix(".csv"), index=False)
print(f"Saved: {output_path}")

In [ ]:
# pull the latent z-columns and the transfer cluster labels from aou_assignments
z_cols = [c for c in aou_assignments.columns if c.startswith("z_")]
Z = aou_assignments[z_cols].to_numpy()
labels = aou_assignments["cluster"].to_numpy()

n_clusters = len(np.unique(labels))
if n_clusters >= 2 and len(Z) > n_clusters:
    sil = silhouette_score(Z, labels)
    print(f"Transfer silhouette (JHU latent space): {sil:.4f}")
    print(f"  computed over {len(Z)} patients, {n_clusters} clusters")
else:
    print("Cannot compute silhouette:", n_clusters, "clusters")

In [ ]:
# run on JHU 
jhu_feats, jhu_mask = encode_measurements(raw_jhu, jhu_scaler, clip=4.0)
jhu_feats, jhu_mask = merge_unit_splits(jhu_feats, jhu_mask, jhu_metadata)
jhu_feats = jhu_feats.reindex(columns=jhu_feature_order)
jhu_mask = jhu_mask.reindex(columns=jhu_feature_order, fill_value=0)
jhu_mask = jhu_mask.where(jhu_feats.notna(), 0).fillna(0)
jhu_feats = jhu_feats.fillna(0)

X_jhu = jhu_feats.to_numpy(dtype="float32")
M_jhu = jhu_mask.to_numpy(dtype="float32")
X_jhu_in = neutral_fill(X_jhu, M_jhu, jhu_band_rate)

jhu_assignments = predict_clusters(model, X_jhu_in, index=raw_jhu.index, device=DEVICE)

z_cols = [c for c in aou_assignments.columns if c.startswith("z_")]
Z_jhu = jhu_assignments[z_cols].to_numpy()
labels_jhu = jhu_assignments["cluster"].to_numpy()
Z_aou = aou_assignments[z_cols].to_numpy()
labels_aou = aou_assignments["cluster"].to_numpy()

print(f"JHU patients: {len(Z_jhu)}   AoU patients: {len(Z_aou)}")

# centroids
K = len(np.unique(labels_jhu))
centroids = np.array([Z_jhu[labels_jhu == k].mean(axis=0) for k in range(K)])

def dist_to_assigned_centroid(Z, labels, centroids):
    return np.linalg.norm(Z - centroids[labels], axis=1)

d_jhu = dist_to_assigned_centroid(Z_jhu, labels_jhu, centroids)
d_aou = dist_to_assigned_centroid(Z_aou, labels_aou, centroids)

print("Distance to assigned centroid (JHU latent space)")
print(f"JHU: mean={d_jhu.mean():.3f}  median={np.median(d_jhu):.3f}")
print(f"AoU: mean={d_aou.mean():.3f}  median={np.median(d_aou):.3f}")

from scipy.stats import mannwhitneyu
u_stat, p_val = mannwhitneyu(d_jhu, d_aou, alternative="two-sided")
print(f"Mann-Whitney U test (JHU vs AoU distances): p = {p_val:.4g}")

print("\nPer-cluster distance comparison:")
for k in range(K):
    dj, da = d_jhu[labels_jhu == k], d_aou[labels_aou == k]
    print(f"  cluster {k}: JHU mean={dj.mean():.3f} (n={len(dj)})   "
          f"AoU mean={da.mean():.3f} (n={len(da)})")

# Davies-Bouldin index on the AoU-transferred partition
db_aou = davies_bouldin_score(Z_aou, labels_aou)
db_jhu = davies_bouldin_score(Z_jhu, labels_jhu)
print(f"\nDavies-Bouldin -- JHU: {db_jhu:.4f}   AoU (transfer): {db_aou:.4f}")

# Cluster-size proportions
jhu_counts = pd.Series(labels_jhu).value_counts().sort_index()
aou_counts = pd.Series(labels_aou).value_counts().sort_index()
size_table = pd.DataFrame({"JHU": jhu_counts, "AoU_transfer": aou_counts}).fillna(0)
print("\nCluster size comparison:")
print(size_table)
print("JHU proportions:", (jhu_counts / jhu_counts.sum()).round(3).to_dict())
print("AoU proportions:", (aou_counts / aou_counts.sum()).round(3).to_dict())

chi2, p_chi2, dof, _ = chi2_contingency(size_table.T.values)
print(f"Chi-square test of proportion homogeneity: chi2={chi2:.3f}, p={p_chi2:.4g}")